<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [5]:
# Aquí importo las herramientas para leer el corpus y mostrar las tablas.
import re
from collections import Counter
from math import log, log1p

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg

# Aquí cargo los 18 textos de Gutenberg que usé en la práctica anterior.
try:
    ids_documentos = gutenberg.fileids()
except LookupError:
    nltk.download("gutenberg", raise_on_error=True)
    ids_documentos = gutenberg.fileids()

# Aquí reutilizo la tokenización anterior: minúsculas y apóstrofos internos.
TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", flags=re.UNICODE)

def tokenizar(texto):
    return TOKEN_RE.findall(texto.lower().replace("\u2019", "'"))

# Aquí guardo el texto original para los fragmentos y los tokens para el cálculo.
textos_originales = {}
tokens_documentos = {}
titulos = {}
filas_corpus = []
for id_doc in ids_documentos:
    texto = gutenberg.raw(id_doc)
    textos_originales[id_doc] = texto
    tokens_documentos[id_doc] = tokenizar(texto)
    # Aquí extraigo el título del encabezado y quito el autor cuando aparece.
    encabezado = texto.lstrip().splitlines()[0].strip().strip("[]")
    titulos[id_doc] = encabezado.split(" by ", 1)[0].rstrip(",")
    filas_corpus.append({
        "Identificador de documento": id_doc,
        "Título": titulos[id_doc],
        "Número de términos únicos": len(set(tokens_documentos[id_doc])),
    })

# Aquí resumo el tamaño del corpus y la longitud de cada texto en tokens.
n_documentos = len(ids_documentos)
longitudes = np.array([len(tokens_documentos[id_doc]) for id_doc in ids_documentos])
vocabulario_corpus = set().union(*tokens_documentos.values())
display(pd.DataFrame(filas_corpus))
display(pd.DataFrame([{
    "Documentos": n_documentos,
    "Tamaño del vocabulario": len(vocabulario_corpus),
    "Longitud mínima (tokens)": longitudes.min(),
    "Longitud media (tokens)": round(longitudes.mean(), 2),
    "Longitud máxima (tokens)": longitudes.max(),
}]))

,Identificador de documento,Título,Número de términos únicos
0,austen-emma.txt,Emma,7211
1,austen-persuasion.txt,Persuasion,5861
2,austen-sense.txt,Sense and Sensibility,6431
3,bible-kjv.txt,The King James Bible,12965
4,blake-poems.txt,Poems,1544
5,bryant-stories.txt,Stories to Tell to Children,3915
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1549
7,carroll-alice.txt,Alice's Adventures in Wonderland,2630
8,chesterton-ball.txt,The Ball and The Cross,8341
9,chesterton-brown.txt,The Wisdom of Father Brown,7884


,Documentos,Tamaño del vocabulario,Longitud mínima (tokens),Longitud media (tokens),Longitud máxima (tokens)
0,18,43825,6839,121225.78,852256


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [12]:
# Aquí calculo TF contando todas las apariciones de cada término.
def calcular_tf(tokens):
    return Counter(tokens)

# Aquí sumo uno al DF por cada documento que contiene el término.
def calcular_df(conteos_documentos):
    frecuencias = Counter()
    for conteos in conteos_documentos.values():
        frecuencias.update(conteos.keys())
    return frecuencias

# Aquí uso el logaritmo natural para el IDF sin suavizado.
def calcular_idf(df, total_documentos):
    return log(total_documentos / df) + 1

# Aquí multiplico la frecuencia del término por su IDF.
def calcular_peso(tf, idf):
    return tf * idf

# Aquí obtengo los conteos de todos los documentos antes de hacer los ejemplos.
tf_por_documento = {
    id_doc: calcular_tf(tokens_documentos[id_doc])
    for id_doc in ids_documentos
}
df_corpus = calcular_df(tf_por_documento)

# Aquí muestro seis pesos TF-IDF con tres términos y dos novelas de Austen.
ejemplos = []
for id_doc in ("austen-emma.txt", "austen-sense.txt"):
    for termino in ("marriage", "family", "fortune"):
        tf = tf_por_documento[id_doc][termino]
        df = df_corpus[termino]
        idf = calcular_idf(df, n_documentos)
        ejemplos.append({
            "Documento": id_doc,
            "Término": termino,
            "TF": tf,
            "DF": df,
            "IDF": idf,
            "TF-IDF": calcular_peso(tf, idf),
        })

display(pd.DataFrame(ejemplos))

,Documento,Término,TF,DF,IDF,TF-IDF
0,austen-emma.txt,marriage,35,12,1.405465,49.191279
1,austen-emma.txt,family,77,13,1.325422,102.057525
2,austen-emma.txt,fortune,39,12,1.405465,54.813139
3,austen-sense.txt,marriage,44,12,1.405465,61.840465
4,austen-sense.txt,family,83,13,1.325422,110.010059
5,austen-sense.txt,fortune,38,12,1.405465,53.407674


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [15]:
# Aquí ordeno el vocabulario para que cada término tenga una columna fija.
vocabulario = sorted(vocabulario_corpus)
columnas = {termino: i for i, termino in enumerate(vocabulario)}

# Aquí reutilizo los conteos TF y DF de la celda anterior y calculo el IDF del corpus.
idf_corpus = {
    termino: calcular_idf(df_corpus[termino], n_documentos)
    for termino in vocabulario
}

# Aquí armo la matriz: las filas son documentos y las columnas son términos.
# Empiezo con ceros para que los términos ausentes queden con peso cero.
tfidf_manual = np.zeros((n_documentos, len(vocabulario)))
for fila, id_doc in enumerate(ids_documentos):
    for termino, tf in tf_por_documento[id_doc].items():
        columna = columnas[termino]
        tfidf_manual[fila, columna] = calcular_peso(tf, idf_corpus[termino])

print(f"Matriz TF-IDF manual: {tfidf_manual.shape[0]} documentos × "
      f"{tfidf_manual.shape[1]:,} términos.")

Matriz TF-IDF manual: 18 documentos × 43,825 términos.


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Aquí le paso mi tokenizador y el orden de columnas de la matriz manual.
vectorizador = TfidfVectorizer(
    analyzer=tokenizar,
    vocabulary=columnas,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
tfidf_sklearn = vectorizador.fit_transform(
    [textos_originales[id_doc] for id_doc in ids_documentos]
)

# Aquí comparo todos los pesos y verifico la mayor diferencia absoluta.
TOLERANCIA = 1e-10
tfidf_sklearn_denso = tfidf_sklearn.toarray()
diferencia_tfidf = np.max(np.abs(tfidf_manual - tfidf_sklearn_denso))
print(f"Máxima diferencia absoluta de TF-IDF: {diferencia_tfidf:.3e}")
print(f"Tolerancia absoluta: {TOLERANCIA:.0e}")
np.testing.assert_allclose(
    tfidf_manual, tfidf_sklearn_denso, atol=TOLERANCIA, rtol=0
)

Máxima diferencia absoluta de TF-IDF: 0.000e+00
Tolerancia absoluta: 1e-10


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

# Aquí defino cuatro consultas y anoto qué espera encontrar el usuario.
consultas = {
    "marriage family fortune": "Novelas sobre el matrimonio y la riqueza familiar.",
    "whale ship captain": "Relatos de viajes en barco y caza de ballenas.",
    "king crown battle": "Obras sobre reyes, disputas por la corona y batallas.",
    "faith prayer god": "Textos religiosos sobre la fe y la oración.",
}
tokens_consultas = {consulta: tokenizar(consulta) for consulta in consultas}
display(pd.DataFrame([
    {
        "Consulta": consulta,
        "Necesidad de información": necesidad,
        "Tokens": ", ".join(tokens_consultas[consulta]),
    }
    for consulta, necesidad in consultas.items()
]))

# Aquí represento cada consulta con el vocabulario y los IDF ya calculados.
def vector_consulta(tokens):
    vector = np.zeros(len(vocabulario))
    for termino, tf in calcular_tf(tokens).items():
        if termino in columnas:
            vector[columnas[termino]] = calcular_peso(tf, idf_corpus[termino])
    return vector

consultas_manual = np.array([
    vector_consulta(tokens_consultas[consulta]) for consulta in consultas
])
# Aquí uso transform porque el vectorizador ya fue ajustado con el corpus.
consultas_sklearn = vectorizador.transform(list(consultas))
np.testing.assert_allclose(
    consultas_manual, consultas_sklearn.toarray(), atol=TOLERANCIA, rtol=0
)

# Aquí calculo los productos punto y divido cada uno por el producto de las normas.
def coseno_manual(vectores_consulta, matriz_documentos):
    productos_punto = vectores_consulta @ matriz_documentos.T
    normas_consulta = np.linalg.norm(vectores_consulta, axis=1)
    normas_documentos = np.linalg.norm(matriz_documentos, axis=1)
    denominador = normas_consulta[:, None] * normas_documentos[None, :]
    # Aquí dejo similitud cero cuando una consulta o documento tiene norma cero.
    return np.divide(
        productos_punto, denominador,
        out=np.zeros_like(productos_punto), where=denominador != 0
    )

puntuaciones_manual = coseno_manual(consultas_manual, tfidf_manual)
puntuaciones_sklearn = cosine_similarity(consultas_sklearn, tfidf_sklearn)
np.testing.assert_allclose(
    puntuaciones_manual, puntuaciones_sklearn, atol=TOLERANCIA, rtol=0
)
diferencia_coseno = np.max(np.abs(puntuaciones_manual - puntuaciones_sklearn))
print(f"Máxima diferencia absoluta de coseno: {diferencia_coseno:.3e}")

# Aquí desgloso la primera consulta con el documento de mayor puntuación.
primera_consulta = next(iter(consultas))
indice_doc = int(puntuaciones_manual[0].argmax())
vector_q = consultas_manual[0]
vector_d = tfidf_manual[indice_doc]
comunes = np.flatnonzero((vector_q > 0) & (vector_d > 0))
print(f"Consulta: {primera_consulta!r} | Documento: {ids_documentos[indice_doc]}")
display(pd.DataFrame({
    "Términos compartidos": [vocabulario[i] for i in comunes],
    "Peso en consulta": vector_q[comunes],
    "Peso en documento": vector_d[comunes],
    "Productos parciales": vector_q[comunes] * vector_d[comunes],
}))
display(pd.DataFrame([{
    "Producto punto": np.dot(vector_q, vector_d),
    "Norma de la consulta": np.linalg.norm(vector_q),
    "Norma del documento": np.linalg.norm(vector_d),
    "Similitud": puntuaciones_manual[0, indice_doc],
}]))

,Consulta,Necesidad de información,Tokens
0,marriage family fortune,Novelas sobre el matrimonio y la riqueza famil...,"marriage, family, fortune"
1,whale ship captain,Relatos de viajes en barco y caza de ballenas.,"whale, ship, captain"
2,king crown battle,"Obras sobre reyes, disputas por la corona y ba...","king, crown, battle"
3,faith prayer god,Textos religiosos sobre la fe y la oración.,"faith, prayer, god"


Máxima diferencia absoluta de coseno: 2.276e-15
Consulta: 'marriage family fortune' | Documento: austen-persuasion.txt


,Términos compartidos,Peso en consulta,Peso en documento,Productos parciales
0,family,1.325422,106.033792,140.539563
1,fortune,1.405465,40.758488,57.284633
2,marriage,1.405465,37.947558,53.333969


,Producto punto,Norma de la consulta,Norma del documento,Similitud
0,251.158165,2.389018,8000.438152,0.013141


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [36]:
# Aquí busco un fragmento alrededor de un término compartido con la consulta.
def obtener_fragmento(id_doc, tokens_consulta, radio=80):
    compartidos = set(tokens_consulta).intersection(tf_por_documento[id_doc])
    if not compartidos:
        return "Sin términos compartidos."

    # Aquí elijo el término de mayor IDF para mostrar un fragmento específico.
    termino_clave = max(sorted(compartidos), key=idf_corpus.get)
    texto = textos_originales[id_doc]
    for coincidencia in TOKEN_RE.finditer(texto.replace("\u2019", "'")):
        if coincidencia.group().lower() == termino_clave:
            inicio = max(0, coincidencia.start() - radio)
            fin = min(len(texto), coincidencia.end() + radio)
            fragmento = " ".join(texto[inicio:fin].split())
            return f"…{fragmento}…"

# Aquí ordeno de mayor a menor puntuación y uso el ID para desempatar.
def top_cinco(puntuaciones, tokens_consulta):
    orden = sorted(
        range(n_documentos),
        key=lambda i: (-puntuaciones[i], ids_documentos[i]),
    )
    filas = []
    for posicion, i in enumerate(orden[:5], start=1):
        id_doc = ids_documentos[i]
        filas.append({
            "Posición": posicion,
            "Identificador": id_doc,
            "Título": titulos[id_doc],
            "Puntuación": puntuaciones[i],
            "Fragmento": obtener_fragmento(id_doc, tokens_consulta),
        })
    return pd.DataFrame(filas)

# Aquí uso BM25 con k1 para la saturación de TF y b para ajustar por longitud.
def puntuar_bm25(tokens_consulta, k1=1.2, b=0.75):
    puntuaciones = np.zeros(n_documentos)
    ajuste_longitud = k1 * (1 - b + b * longitudes / longitudes.mean())
    for termino in dict.fromkeys(tokens_consulta):
        df = df_corpus[termino]
        if df == 0:
            continue
        # Aquí calculo el IDF de la variante de BM25 que utiliza Apache Lucene.
        # https://lucene.apache.org/core/10_5_2/core/org/apache/lucene/search/similarities/BM25Similarity.html
        idf_bm25 = log1p((n_documentos - df + 0.5) / (df + 0.5))
        tf = np.array([tf_por_documento[id_doc][termino] for id_doc in ids_documentos])
        puntuaciones += idf_bm25 * (tf * (k1 + 1)) / (tf + ajuste_longitud)
    return puntuaciones

# Aquí muestro los top-5 y cuento cuántos documentos coinciden entre ambos métodos.
print("Las puntuaciones TF-IDF manuales y de scikit-learn coinciden dentro de la tolerancia.")
for i, consulta in enumerate(consultas):
    tokens = tokens_consultas[consulta]
    ranking_tfidf = top_cinco(puntuaciones_manual[i], tokens)
    ranking_bm25 = top_cinco(puntuar_bm25(tokens), tokens)

    print(f"\nConsulta: {consulta!r}")
    with pd.option_context("display.max_colwidth", None):
        print("Top 5 TF-IDF con similitud coseno")
        display(ranking_tfidf)
        print("Top 5 BM25")
        display(ranking_bm25)

    ids_comunes = set(ranking_tfidf["Identificador"]) & set(ranking_bm25["Identificador"])
    print(f"Documentos en común entre ambos top 5: {len(ids_comunes)} de 5.")

Las puntuaciones TF-IDF manuales y de scikit-learn coinciden dentro de la tolerancia.

Consulta: 'marriage family fortune'
Top 5 TF-IDF con similitud coseno


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-persuasion.txt,Persuasion,0.013141,"…merely connected herself with an old country family of respectability and large fortune, and had therefore given all the honour and received none: Elizabeth would, one…"
1,2,austen-sense.txt,Sense and Sensibility,0.011320,"…ghters. The son, a steady respectable young man, was amply provided for by the fortune of his mother, which had been large, and half of which devolved on him on his c…"
2,3,austen-emma.txt,Emma,0.007774,"…piness for her friend. Mr. Weston was a man of unexceptionable character, easy fortune, suitable age, and pleasant manners; and there was some satisfaction in conside…"
3,4,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.004853,"…of her fauour? Guil. Faith, her priuates, we Ham. In the secret parts of Fortune? Oh, most true: she is a Strumpet. What's the newes? Rosin. None my Lord; but…"
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.004641,"…matics and economics at Cambridge. He left me practically all his pretty large fortune; and I am sure he meant it in contempt. ""Arthur, you may say, might well…"


Top 5 BM25


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-sense.txt,Sense and Sensibility,2.531286,"…ghters. The son, a steady respectable young man, was amply provided for by the fortune of his mother, which had been large, and half of which devolved on him on his c…"
1,2,austen-persuasion.txt,Persuasion,2.526870,"…merely connected herself with an old country family of respectability and large fortune, and had therefore given all the honour and received none: Elizabeth would, one…"
2,3,austen-emma.txt,Emma,2.508096,"…piness for her friend. Mr. Weston was a man of unexceptionable character, easy fortune, suitable age, and pleasant manners; and there was some satisfaction in conside…"
3,4,chesterton-thursday.txt,The Man Who Was Thursday,2.174590,"…rmed. ""Yes, the thing really happened to me,"" he said hastily. ""I had the good fortune to fall into conversation with a detective who took me, thanks to my hat, for a…"
4,5,whitman-leaves.txt,Leaves of Grass,2.143200,"…ere is hut one theme for ever-enduring bards? And that is the theme of War, the fortune of battles, The making of perfect soldiers. Be it so, then I answer'd, I too h…"


Documentos en común entre ambos top 5: 3 de 5.

Consulta: 'whale ship captain'
Top 5 TF-IDF con similitud coseno


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.101403,"…""While you take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter…"
1,2,austen-persuasion.txt,Persuasion,0.036574,"…th, the former curate of Monkford, however suspicious appearances may be, but a Captain Frederick Wentworth, his brother, who being made commander in consequence of th…"
2,3,bryant-stories.txt,Stories to Tell to Children,0.005477,"…t, unless somebody picked him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on the sa…"
3,4,whitman-leaves.txt,Leaves of Grass,0.005000,"…s the king-pin, he heaves down with a strong arm, The mate stands braced in the whale-boat, lance and harpoon are ready, The duck-shooter walks by silent and cautiou…"
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.003710,"…at him, and seemed suddenly to be listening with great attention. The brigand captain took the large paper from the attendant brigand and, glancing over it, continue…"


... (7 outputs left out)

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,bible-kjv.txt,The King James Bible,0.025741,"…build thee an house: therefore hath thy servant found in his heart to pray this prayer unto thee. 7:28 And now, O Lord GOD, thou art that God, and thy words be true,…"
1,2,milton-paradise.txt,Paradise Lost,0.024124,"…suffice, and soften stony hearts To pray, repent, and bring obedience due. To prayer, repentance, and obedience due, Though but endeavour'd with sincere intent, M…"
2,3,chesterton-ball.txt,The Ball and The Cross,0.010292,"…and took off their hats, coats, waistcoats, and boots. Evan said a short Latin prayer to himself, during which Turnbull made something of a parade of lighting a ciga…"
3,4,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.008762,"…now? Whereto serues mercy, But to confront the visage of Offence? And what's in Prayer, but this two-fold force, To be fore-stalled ere we come to fall, Or pardon'd b…"
4,5,blake-poems.txt,Poems,0.008475,"…child. I a child, and thou a lamb, We are called by His name. Little Lamb, God bless thee! Little Lamb, God bless thee! THE LITTLE BLACK BOY My mot…"


Top 5 BM25


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,bible-kjv.txt,The King James Bible,1.976082,"…build thee an house: therefore hath thy servant found in his heart to pray this prayer unto thee. 7:28 And now, O Lord GOD, thou art that God, and thy words be true,…"
1,2,milton-paradise.txt,Paradise Lost,1.881444,"…suffice, and soften stony hearts To pray, repent, and bring obedience due. To prayer, repentance, and obedience due, Though but endeavour'd with sincere intent, M…"
2,3,chesterton-ball.txt,The Ball and The Cross,1.859655,"…and took off their hats, coats, waistcoats, and boots. Evan said a short Latin prayer to himself, during which Turnbull made something of a parade of lighting a ciga…"
3,4,whitman-leaves.txt,Leaves of Grass,1.847136,"…er I touch or am touch'd from, The scent of these arm-pits aroma finer than prayer, This head more than churches, bibles, and all the creeds. If I worship one th…"
4,5,bryant-stories.txt,Stories to Tell to Children,1.782285,"…ers came back they found their king kneeling by his throne, absorbed in silent prayer. THE JEALOUS COURTIERS I wonder if you have ever heard the anec…"


Documentos en común entre ambos top 5: 3 de 5.


### Conclusión

Al trabajar con los 18 documentos de Gutenberg, obtuve los mismos pesos TF-IDF con mi implementación y con scikit-learn. La diferencia máxima en la similitud coseno fue de aproximadamente $2{,}28 \times 10^{-15}$. Los rankings de TF-IDF y BM25 compartieron entre dos y tres documentos en el top 5 de cada consulta. Por ejemplo, para `whale ship captain`, ambos métodos ubicaron *Moby Dick* en primer lugar, aunque el resto del ranking varió.

### Fuente de BM25 y uso de IA

Para implementar BM25, consulté la documentación de Apache Lucene. De ahí tomé la fórmula del IDF y los valores $k_1 = 1{,}2$ y $b = 0{,}75$, que controlan el efecto de la frecuencia de los términos y de la longitud de los documentos, respectivamente. También utilicé IA como apoyo para investigar el método y explicar su cálculo paso a paso. [Documentación de BM25Similarity en Apache Lucene](https://lucene.apache.org/core/10_5_2/core/org/apache/lucene/search/similarities/BM25Similarity.html)